<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/Ciencia%20De%20Datos%20Con%20Python/CuadernosColab/CienciaDeDatos_Cap%C3%ADtulo013_Qu%C3%A9_significa_limpiar_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Capítulo 14 — Valores faltantes

## Breve repaso

En el trabajo anterior sobre limpieza de datos comenzamos una nueva etapa del recorrido. Dejamos de pensar en Pandas solo como un conjunto de instrucciones aisladas y empezamos a pensar en el proceso completo de preparación de un dataset.

Vimos que limpiar datos implica diagnosticar problemas, tomar decisiones, aplicar transformaciones y verificar resultados. También presentamos un dataset real de ventas de cafetería y realizamos una primera inspección general para detectar señales de posibles problemas.

En este capítulo vamos a concentrarnos en uno de los problemas más frecuentes en datasets reales: los valores faltantes.

Un valor faltante aparece cuando una celda no tiene información disponible. Esto puede ocurrir por muchos motivos: errores de carga, datos que no fueron registrados, campos que no correspondían para cierta observación, problemas al exportar un archivo o decisiones tomadas por el sistema que generó los datos.

El objetivo de este capítulo no será eliminar ni completar valores faltantes todavía. Antes de decidir qué hacer con ellos, necesitamos entenderlos. Vamos a aprender a detectarlos, contarlos, calcular su proporción y observar en qué columnas aparecen.

También vamos a distinguir entre valores realmente faltantes, como `NaN`, y valores escritos como texto, como `"UNKNOWN"` o `"ERROR"`, que pueden representar información desconocida o problemas de carga, pero que Pandas no siempre interpreta automáticamente como faltantes.

Al finalizar este capítulo deberías poder:

- Explicar qué significa que falte un dato.
- Reconocer cómo representa Pandas los valores faltantes.
- Detectar valores faltantes con `isna()`.
- Contar valores faltantes por columna.
- Calcular porcentajes de valores faltantes.
- Ver filas que contienen datos faltantes.
- Distinguir entre `NaN` y valores textuales como `"UNKNOWN"` o `"ERROR"`.
- Construir un primer diagnóstico de faltantes sin tomar decisiones apresuradas.

## Dataset de trabajo

En este capítulo vamos a seguir trabajando con el dataset **Cafe Sales — Dirty Data for Cleaning Training**.

Cada fila representa una transacción de venta en una cafetería. Las columnas describen información como el producto vendido, la cantidad, el precio unitario, el total gastado, el método de pago, la ubicación y la fecha de la transacción.

Este dataset es especialmente útil para estudiar valores faltantes porque contiene datos incompletos y valores problemáticos que necesitaremos diagnosticar antes de decidir cómo tratarlos.

Como cada capítulo debe poder ejecutarse de manera independiente, vamos a descargar y cargar nuevamente el dataset.

In [ ]:
import kagglehub
import pandas as pd
import os

ruta_dataset = kagglehub.dataset_download(
    "ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training"
)

archivos = os.listdir(ruta_dataset)

archivos_csv = [
    archivo for archivo in archivos
    if archivo.endswith(".csv")
]

ruta_csv = os.path.join(ruta_dataset, archivos_csv[0])

df = pd.read_csv(ruta_csv)

df.head()

Using Colab cache for faster access to the 'cafe-sales-dirty-data-for-cleaning-training' dataset.


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


La salida de `head()` nos permite confirmar que el archivo fue cargado correctamente.

A partir de este punto vamos a concentrarnos en una pregunta específica: ¿qué datos faltan en este dataset?

Para responderla no alcanza con mirar algunas filas. Necesitamos usar herramientas que nos permitan detectar y contar valores faltantes en todo el `DataFrame`.

## Qué significa que falte un dato

Un valor faltante aparece cuando una celda del dataset no contiene información disponible.

Esto puede parecer un problema simple, pero no siempre significa lo mismo. Un dato puede faltar por muchas razones. Tal vez la información no fue registrada, tal vez hubo un error al cargar el archivo, tal vez el campo no correspondía para esa transacción, o tal vez el dato existe pero fue escrito con un valor especial que Pandas no reconoce automáticamente como faltante.

Por ejemplo, en un dataset de ventas de cafetería no es lo mismo que falte el método de pago, que falte el producto vendido o que falte el total gastado. Cada columna cumple una función distinta dentro del análisis, y por eso la ausencia de datos puede tener distintos niveles de gravedad.

Si falta el método de pago, quizás todavía podamos conservar la transacción para analizar productos vendidos o importes. En cambio, si falta la cantidad o el precio unitario, tal vez no podamos calcular correctamente el total de la venta. Si falta la fecha, podríamos perder la posibilidad de analizar la evolución temporal de las ventas.

Por eso, antes de eliminar o completar valores faltantes, necesitamos entender qué representan.

Una buena pregunta inicial no es:

```text
¿Cómo saco los valores faltantes?
```

sino:

```text
¿Qué significa que falte este dato en esta columna?
```

Ese cambio de enfoque es importante. Los valores faltantes no son solamente un problema técnico. También son una señal sobre la calidad, el origen y la confiabilidad del dataset.

## Cómo representa Pandas los valores faltantes

En Pandas, los valores faltantes suelen representarse como `NaN`.

`NaN` significa *Not a Number*, pero en Pandas se usa de manera más amplia para indicar ausencia de información. Puede aparecer en columnas numéricas, columnas de texto, fechas u otros tipos de datos.

Cuando vemos `NaN` en una celda, significa que Pandas reconoce que allí falta un valor.

Es importante distinguir entre un valor faltante real y otros valores que pueden parecer faltantes, pero que Pandas interpreta como texto común.

Por ejemplo, estos valores pueden aparecer en un dataset:

```text
NaN
UNKNOWN
ERROR
N/A
Sin dato
```

Para Pandas, `NaN` es un valor faltante reconocido. En cambio, valores como `"UNKNOWN"` o `"ERROR"` pueden estar escritos como texto. Eso significa que Pandas no necesariamente los va a contar como faltantes cuando usemos herramientas como `isna()`.

Esta diferencia será importante en este capítulo. Primero vamos a detectar los valores que Pandas reconoce como faltantes. Luego vamos a revisar valores textuales que pueden representar información desconocida o errores de carga.

## Detectar valores faltantes con isna()

Para detectar valores faltantes en Pandas podemos usar `isna()`.

Este método revisa cada celda del `DataFrame` y devuelve `True` cuando encuentra un valor faltante, y `False` cuando la celda contiene algún valor.

Veamos qué ocurre al aplicarlo sobre el dataset.

In [ ]:
df.isna()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...
9995,False,False,False,False,False,True,False,False
9996,False,True,False,True,False,False,True,False
9997,False,False,False,False,False,False,True,False
9998,False,False,False,True,False,False,True,False


La salida tiene la misma forma que el `DataFrame` original, pero en lugar de mostrar los datos muestra valores booleanos.

Cada `True` indica que en esa posición hay un valor faltante. Cada `False` indica que en esa posición hay un valor presente.

Esta vista puede ser útil para entender la idea, pero en un dataset grande no resulta práctica para leer directamente. Si tenemos muchas filas y columnas, una tabla llena de `True` y `False` no alcanza para saber rápidamente dónde está el problema.

Por eso, el siguiente paso será contar cuántos valores faltantes hay en cada columna.

## Contar valores faltantes por columna

Para que la detección de valores faltantes sea más útil, podemos combinar `isna()` con `sum()`.

Primero, `isna()` identifica las celdas faltantes. Luego, `sum()` cuenta cuántos valores `True` hay en cada columna.

En Pandas, los valores `True` se cuentan como `1` y los valores `False` como `0`. Por eso, al sumar una máscara booleana obtenemos la cantidad de valores faltantes.

In [ ]:
df.isna().sum()

,0
Transaction ID,0
Item,333
Quantity,138
Price Per Unit,179
Total Spent,173
Payment Method,2579
Location,3265
Transaction Date,159


La salida muestra cuántos valores faltantes hay en cada columna.

Este conteo nos permite identificar rápidamente qué columnas tienen datos incompletos y cuáles parecen estar completas.

Sin embargo, todavía necesitamos interpretar esos números. Una columna con 100 valores faltantes puede parecer problemática, pero su importancia depende del tamaño total del dataset. No es lo mismo tener 100 faltantes en un dataset de 200 filas que en un dataset de 100.000 filas.

Por eso, además de contar valores faltantes, conviene calcular qué porcentaje representan.

## Calcular porcentajes de valores faltantes

El conteo absoluto de valores faltantes es útil, pero no siempre alcanza para interpretar la magnitud del problema.

Para comprender mejor el impacto de los faltantes, conviene calcular qué porcentaje representan sobre el total de filas del dataset.

Podemos hacerlo usando `isna().mean()`. Como `True` se interpreta como `1` y `False` como `0`, el promedio de una máscara booleana indica la proporción de valores faltantes.

In [ ]:
porcentaje_faltantes = df.isna().mean() * 100

porcentaje_faltantes

,0
Transaction ID,0.00
Item,3.33
Quantity,1.38
Price Per Unit,1.79
Total Spent,1.73
Payment Method,25.79
Location,32.65
Transaction Date,1.59


La salida muestra el porcentaje de valores faltantes en cada columna.

Por ejemplo, si una columna muestra un valor cercano a `5`, significa que aproximadamente el 5% de sus filas tienen valores faltantes. Si una columna muestra `0`, significa que Pandas no detectó valores faltantes en esa columna.

También podemos combinar el conteo y el porcentaje en una misma tabla para leer el diagnóstico con mayor claridad.

In [ ]:
diagnostico_faltantes = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "porcentaje": df.isna().mean() * 100
})

diagnostico_faltantes

,faltantes,porcentaje
Transaction ID,0,0.00
Item,333,3.33
Quantity,138,1.38
Price Per Unit,179,1.79
Total Spent,173,1.73
Payment Method,2579,25.79
Location,3265,32.65
Transaction Date,159,1.59


Esta tabla resume, para cada columna, cuántos valores faltantes hay y qué porcentaje representan.

Este tipo de diagnóstico es muy útil porque nos permite priorizar. Una columna con pocos faltantes puede requerir una estrategia diferente de una columna con muchos faltantes. Además, no todas las columnas tienen la misma importancia para el análisis.

Por ahora seguimos en la etapa de diagnóstico. Todavía no vamos a eliminar ni completar valores faltantes.

## Ordenar el diagnóstico de faltantes

Cuando un dataset tiene muchas columnas, la tabla de diagnóstico puede ser más fácil de leer si la ordenamos según la cantidad de valores faltantes.

Podemos ordenar la tabla `diagnostico_faltantes` usando `sort_values()`.

In [ ]:
diagnostico_faltantes.sort_values("faltantes", ascending=False)

,faltantes,porcentaje
Location,3265,32.65
Payment Method,2579,25.79
Item,333,3.33
Price Per Unit,179,1.79
Total Spent,173,1.73
Transaction Date,159,1.59
Quantity,138,1.38
Transaction ID,0,0.00


Ahora las columnas con más valores faltantes aparecen primero.

Esta forma de presentación ayuda a identificar rápidamente dónde se concentra el problema. En una limpieza real, muchas veces conviene empezar revisando las columnas con mayor cantidad de faltantes o aquellas que son más importantes para el análisis.

También podemos ordenar por porcentaje:

In [ ]:
diagnostico_faltantes.sort_values("porcentaje", ascending=False)

,faltantes,porcentaje
Location,3265,32.65
Payment Method,2579,25.79
Item,333,3.33
Price Per Unit,179,1.79
Total Spent,173,1.73
Transaction Date,159,1.59
Quantity,138,1.38
Transaction ID,0,0.00


En este caso, el resultado es similar porque el porcentaje se calcula a partir del mismo total de filas. Sin embargo, incluir el porcentaje facilita la interpretación.

Un número absoluto puede parecer grande o pequeño según el tamaño del dataset. En cambio, el porcentaje permite evaluar mejor el peso relativo del problema.

Por ejemplo, una columna con 500 valores faltantes puede parecer muy problemática. Pero si el dataset tiene 100.000 filas, esos 500 faltantes representan solo el 0,5%. En cambio, si el dataset tiene 1.000 filas, representan el 50%.

Por eso, cuando analizamos faltantes, conviene mirar siempre dos cosas:

```text
cantidad de valores faltantes
porcentaje sobre el total de filas
```

Ambas medidas se complementan y ayudan a tomar mejores decisiones.


## Ver filas con valores faltantes

Después de contar los valores faltantes por columna, puede ser útil observar algunas filas donde esos faltantes aparecen.

Esto nos permite pasar del diagnóstico general a una revisión más concreta. No alcanza con saber que una columna tiene valores faltantes; también conviene mirar ejemplos reales para entender cómo aparecen dentro del dataset.

Podemos construir una máscara que indique qué filas tienen al menos un valor faltante.

In [ ]:
filas_con_faltantes = df.isna().any(axis=1)

filas_con_faltantes

,0
0,False
1,False
2,False
3,False
4,False
...,...
9995,True
9996,True
9997,True
9998,True


La instrucción `df.isna()` detecta los valores faltantes en todo el `DataFrame`.

Luego usamos `.any(axis=1)` para preguntar, fila por fila, si aparece al menos un valor faltante.

El parámetro `axis=1` indica que queremos revisar horizontalmente, es decir, a lo largo de las columnas de cada fila.

El resultado es una máscara booleana: `True` para las filas que tienen al menos un valor faltante, y `False` para las filas que no tienen faltantes detectados por Pandas.

Ahora podemos usar esa máscara para ver solamente las filas incompletas.

In [ ]:
df[filas_con_faltantes].head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31
13,TXN_9437049,Cookie,5,1.0,5.0,NaN,Takeaway,2023-06-01
14,TXN_8915701,ERROR,2,1.5,3.0,NaN,In-store,2023-03-21
16,TXN_3765707,Sandwich,1,4.0,4.0,NaN,NaN,2023-06-10
23,TXN_2616390,Sandwich,2,4.0,8.0,NaN,NaN,2023-09-18
25,TXN_7958992,Smoothie,3,4.0,NaN,UNKNOWN,UNKNOWN,2023-12-13
28,TXN_8467949,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-11
30,TXN_1736287,NaN,5,2.0,10.0,Digital Wallet,NaN,2023-06-02


Esta vista nos muestra algunos ejemplos de filas con valores faltantes.

Observar casos concretos ayuda a interpretar el problema. Podemos revisar en qué columnas faltan datos, si faltan varios valores en una misma fila o si los faltantes parecen concentrarse en ciertos tipos de registros.

Como el dataset puede tener muchas filas incompletas, usamos `head(10)` para ver solo las primeras diez. Esta muestra no reemplaza el diagnóstico general, pero nos ayuda a comprender mejor cómo se ven los faltantes dentro de la tabla.

## Analizar faltantes en columnas importantes

No todas las columnas tienen la misma importancia para el análisis.

En un dataset de ventas, algunas columnas son centrales porque permiten entender qué se vendió, cuánto se vendió, cuánto costó y cuándo ocurrió la transacción. Otras columnas pueden ser importantes para ciertos análisis, pero no para todos.

Por ejemplo, en este dataset hay columnas especialmente relevantes:

```text
Item
Quantity
Price Per Unit
Total Spent
Transaction Date
```

Si faltan valores en `Item`, no sabemos qué producto se vendió. Si faltan valores en `Quantity` o `Price Per Unit`, se vuelve difícil calcular el importe de la venta. Si falta `Total Spent`, tal vez podamos reconstruirlo usando otras columnas, pero primero necesitamos revisar el caso. Si falta `Transaction Date`, perdemos información temporal.

Podemos observar el diagnóstico solamente para algunas columnas importantes.

In [ ]:
columnas_importantes = [
    "Item",
    "Quantity",
    "Price Per Unit",
    "Total Spent",
    "Transaction Date"
]

diagnostico_faltantes.loc[columnas_importantes]

,faltantes,porcentaje
Item,333,3.33
Quantity,138,1.38
Price Per Unit,179,1.79
Total Spent,173,1.73
Transaction Date,159,1.59


Esta selección nos permite concentrarnos en columnas que probablemente tengan un impacto fuerte sobre el análisis.

La importancia de un valor faltante no depende solamente de cuántos faltantes hay, sino también de qué representa la columna. Un faltante en una columna secundaria puede tener poco impacto. Un faltante en una columna central puede impedir cálculos importantes o limitar las conclusiones posibles.

Por eso, al diagnosticar valores faltantes conviene combinar dos miradas:

```text
magnitud del problema → cuántos faltantes hay
importancia del problema → qué columna está afectada
```

Esa combinación nos ayuda a priorizar qué problemas revisar primero.

## Valores textuales que pueden representar ausencia o error

Hasta ahora usamos `isna()` para detectar valores faltantes reconocidos por Pandas.

Sin embargo, algunos datasets usan palabras o códigos especiales para representar información desconocida o errores de carga. Por ejemplo, pueden aparecer valores como:

```text
UNKNOWN
ERROR
N/A
Sin dato
No informado
```

El problema es que, si esos valores están escritos como texto, Pandas no los cuenta necesariamente como valores faltantes.

Por eso, además de buscar `NaN`, debemos revisar los valores que aparecen dentro de algunas columnas categóricas.


In [ ]:
df["Item"].value_counts(dropna=False)

,count
Item,
Juice,1171
Coffee,1165
Salad,1148
Cake,1139
Sandwich,1131
Smoothie,1096
Cookie,1092
Tea,1089
UNKNOWN,344


In [ ]:
df["Payment Method"].value_counts(dropna=False)

,count
Payment Method,
NaN,2579
Digital Wallet,2291
Credit Card,2273
Cash,2258
ERROR,306
UNKNOWN,293


In [ ]:
df["Location"].value_counts(dropna=False)

,count
Location,
NaN,3265
Takeaway,3022
In-store,3017
ERROR,358
UNKNOWN,338


El parámetro `dropna=False` permite que `value_counts()` incluya también los valores faltantes reales, si los hay.

Al revisar estas salidas, debemos prestar atención a valores como `"UNKNOWN"` o `"ERROR"`. Esos valores no son exactamente iguales a `NaN`, pero pueden indicar que el dato no está disponible, que fue cargado con un código especial o que hubo algún problema en el registro.

Esta diferencia es importante. Si una columna tiene pocos `NaN`, podríamos pensar que está casi completa. Pero si además tiene muchos valores `"UNKNOWN"` o `"ERROR"`, entonces el problema de calidad puede ser mayor de lo que parecía al mirar solo `isna()`.

Por eso, cuando diagnosticamos faltantes, conviene mirar dos cosas:

```text
valores faltantes reconocidos por Pandas
valores textuales que podrían representar ausencia o error
```

Todavía no vamos a decidir cómo tratar esos valores. Por ahora, los estamos identificando para entender mejor el estado real del dataset.

## Comparar NaN con UNKNOWN y ERROR

En este punto ya podemos distinguir dos situaciones diferentes.

Por un lado, están los valores faltantes que Pandas reconoce como `NaN`. Estos aparecen cuando una celda está realmente vacía o cuando Pandas interpreta que allí falta un valor.

Por otro lado, están los valores escritos como texto, como `"UNKNOWN"` o `"ERROR"`. Estos valores pueden indicar ausencia de información o problemas de carga, pero Pandas los trata como textos comunes.

Esto significa que una columna puede parecer bastante completa si miramos solo `isna()`, pero tener muchos valores problemáticos si también contamos `"UNKNOWN"` y `"ERROR"`.

Podemos revisar cuántas veces aparecen esos valores en algunas columnas.

In [ ]:
valores_problematicos = ["UNKNOWN", "ERROR"]

for columna in ["Item", "Payment Method", "Location"]:
    print(f"Columna: {columna}")
    print(df[columna].isin(valores_problematicos).sum())
    print()

Columna: Item
636

Columna: Payment Method
599

Columna: Location
696



En esta celda usamos `isin()` para revisar si los valores de cada columna pertenecen a la lista `["UNKNOWN", "ERROR"]`.

La salida muestra cuántos valores problemáticos aparecen en cada columna revisada.

Esta información complementa el diagnóstico de valores faltantes. No reemplaza a `isna()`, porque `NaN`, `"UNKNOWN"` y `"ERROR"` no son exactamente lo mismo. Pero nos ayuda a ver que el problema de calidad de datos puede ser más amplio que el conteo de celdas vacías.

Podemos construir una pequeña tabla para resumir esta revisión.

In [ ]:
diagnostico_textos_problematicos = pd.DataFrame({
    "UNKNOWN_o_ERROR": {
        columna: df[columna].isin(valores_problematicos).sum()
        for columna in ["Item", "Payment Method", "Location"]
    }
})

diagnostico_textos_problematicos

,UNKNOWN_o_ERROR
Item,636
Payment Method,599
Location,696


Esta tabla nos permite registrar cuántos valores `"UNKNOWN"` o `"ERROR"` aparecen en algunas columnas categóricas.

Más adelante tendremos que decidir qué hacer con esos valores. Podríamos tratarlos como datos faltantes, reemplazarlos por una categoría como `"Desconocido"`, excluirlos de ciertos análisis o revisarlos junto con otras columnas.

Por ahora, lo importante es reconocer que no todos los problemas de ausencia aparecen automáticamente como `NaN`.

## Registrar un diagnóstico de valores faltantes

Después de revisar `NaN`, porcentajes de faltantes y valores textuales como `"UNKNOWN"` o `"ERROR"`, conviene registrar un diagnóstico inicial.

Este diagnóstico no es todavía una limpieza. Es una síntesis del problema.

La idea es reunir evidencia para poder decidir, más adelante, qué estrategia de tratamiento conviene aplicar en cada caso.

Podemos empezar armando una tabla con los faltantes reconocidos por Pandas.

In [ ]:
diagnostico_faltantes = pd.DataFrame({
    "faltantes_nan": df.isna().sum(),
    "porcentaje_nan": df.isna().mean() * 100
})

diagnostico_faltantes.sort_values("faltantes_nan", ascending=False)

,faltantes_nan,porcentaje_nan
Location,3265,32.65
Payment Method,2579,25.79
Item,333,3.33
Price Per Unit,179,1.79
Total Spent,173,1.73
Transaction Date,159,1.59
Quantity,138,1.38
Transaction ID,0,0.00


Esta tabla resume los valores que Pandas reconoce como faltantes.

Pero, como vimos, también pueden existir valores problemáticos escritos como texto. Para algunas columnas categóricas, podemos complementar el diagnóstico contando `"UNKNOWN"` y `"ERROR"`.

In [ ]:
columnas_categoricas_revisar = ["Item", "Payment Method", "Location"]

diagnostico_textos = pd.DataFrame({
    "UNKNOWN_o_ERROR": {
        columna: df[columna].isin(["UNKNOWN", "ERROR"]).sum()
        for columna in columnas_categoricas_revisar
    }
})

diagnostico_textos

,UNKNOWN_o_ERROR
Item,636
Payment Method,599
Location,696


Estas dos tablas nos muestran dos tipos de problemas distintos.

Por un lado, tenemos los faltantes reconocidos como `NaN`. Por otro lado, tenemos valores escritos como texto que pueden representar ausencia de información o errores de carga.

En un análisis real, este diagnóstico debería acompañarse con una interpretación. Por ejemplo:

```text
Algunas columnas tienen valores faltantes reconocidos por Pandas.
Además, ciertas columnas categóricas contienen valores como UNKNOWN o ERROR.
Estos valores no aparecen necesariamente en el conteo de NaN, pero también requieren revisión.
Antes de eliminar o imputar datos, conviene decidir qué representa cada caso.
```

Registrar el diagnóstico nos ayuda a no actuar de manera automática. La limpieza de datos no consiste solamente en aplicar una función, sino en tomar decisiones a partir de evidencia.

## Por qué todavía no vamos a eliminar ni imputar

Después de detectar valores faltantes, puede parecer natural querer resolver el problema de inmediato.

Podríamos pensar en eliminar las filas incompletas, completar valores con algún dato fijo, reemplazar valores desconocidos o reconstruir información faltante a partir de otras columnas.

Sin embargo, todavía no conviene hacerlo.

Antes de eliminar o imputar valores necesitamos comprender mejor el problema. Una decisión apresurada puede afectar el análisis. Por ejemplo, si eliminamos todas las filas con algún valor faltante, podríamos perder muchas transacciones útiles. Si completamos valores sin criterio, podríamos introducir información artificial que modifique los resultados. Si tratamos `"UNKNOWN"` y `"ERROR"` como si fueran exactamente lo mismo, tal vez estemos mezclando ausencia de información con errores de carga.

También debemos recordar que no todas las columnas tienen la misma importancia. Un faltante en `Payment Method` no tiene el mismo impacto que un faltante en `Quantity`, `Price Per Unit` o `Total Spent`. Algunas ausencias pueden limitar ciertos análisis, mientras que otras pueden impedir cálculos centrales.

Por eso, en este capítulo nos concentramos en diagnosticar. El objetivo fue responder preguntas como:

```text
¿Dónde hay valores faltantes?
¿Cuántos hay?
¿Qué porcentaje representan?
¿Qué columnas están afectadas?
¿Existen valores textuales que parecen representar ausencia o error?
¿Qué columnas requieren una revisión más cuidadosa?
```

Una vez que entendemos mejor el problema, estamos en mejores condiciones de decidir una estrategia.

La limpieza de datos responsable no empieza por eliminar o completar, sino por comprender.

## Resumen del capítulo

En este capítulo estudiamos los valores faltantes desde una perspectiva de diagnóstico.

Primero vimos que un valor faltante aparece cuando una celda del dataset no contiene información disponible. También aclaramos que la ausencia de un dato puede tener distintos significados según la columna afectada y el contexto del análisis. No es lo mismo que falte el método de pago, el producto vendido, la cantidad, el precio unitario o la fecha de una transacción.

Luego vimos que Pandas suele representar los valores faltantes como `NaN`. Para detectarlos usamos `isna()`, que devuelve `True` cuando encuentra un valor faltante y `False` cuando la celda contiene algún valor.

Después combinamos `isna()` con `sum()` para contar valores faltantes por columna:

```python
df.isna().sum()
```

También calculamos porcentajes de faltantes usando:

```python
df.isna().mean() * 100
```

Esto nos permitió interpretar mejor la magnitud del problema. Un conteo absoluto puede ser útil, pero el porcentaje ayuda a entender qué peso tienen los faltantes en relación con el tamaño total del dataset.

Más adelante construimos una tabla de diagnóstico:

```python
diagnostico_faltantes = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "porcentaje": df.isna().mean() * 100
})
```

Esa tabla nos permitió observar, columna por columna, cuántos valores faltantes había y qué proporción representaban.

También aprendimos a ver filas con al menos un valor faltante usando:

```python
filas_con_faltantes = df.isna().any(axis=1)

df[filas_con_faltantes].head(10)
```

Este paso fue importante porque nos permitió pasar del resumen general a la observación de casos concretos.

Luego analizamos columnas importantes para el contexto del dataset, como `Item`, `Quantity`, `Price Per Unit`, `Total Spent` y `Transaction Date`. Esto nos ayudó a recordar que no todos los valores faltantes tienen el mismo impacto: la importancia del problema depende tanto de la cantidad de faltantes como del significado de la columna afectada.

También distinguimos entre valores faltantes reconocidos por Pandas y valores textuales que pueden representar ausencia o error, como `"UNKNOWN"` o `"ERROR"`. Estos valores no siempre aparecen en el conteo de `isna()`, porque Pandas puede interpretarlos como texto común.

Por eso revisamos algunas columnas categóricas con:

```python
value_counts(dropna=False)
```

y también contamos apariciones de `"UNKNOWN"` y `"ERROR"` usando `isin()`.

La idea principal de este capítulo fue:

```text
Antes de tratar valores faltantes, necesitamos diagnosticarlos.
```

Detectar faltantes no significa decidir automáticamente qué hacer con ellos. Primero necesitamos saber dónde están, cuántos son, qué porcentaje representan, qué columnas afectan y si existen valores textuales que también deban considerarse problemáticos.

## Próximo paso

El siguiente paso será estudiar estrategias para tratar valores faltantes.

Vamos a analizar cuándo puede tener sentido eliminar filas, cuándo conviene eliminar columnas, cuándo es razonable completar valores faltantes y cuándo podemos reconstruir información usando relaciones internas del dataset.

También vamos a discutir los riesgos de cada decisión. Eliminar datos puede reducir el tamaño del dataset o sesgar el análisis. Imputar valores puede introducir información artificial. Reconstruir datos puede ser útil, pero solo si la relación entre columnas es confiable.

El objetivo será pasar del diagnóstico a la toma de decisiones.